# Clase 179 — ANOVA de una y dos vías

ANOVA compara medias de ≥ 3 grupos sin inflar α (a diferencia de hacer t-tests todos-contra-todos). Vemos `f_oneway`, el supuesto de homocedasticidad (Levene), post-hoc **Tukey HSD**, ANOVA de dos vías con **interacción** vía OLS y el effect size **η²**.

Requiere: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`.

## 1. ANOVA one-way

`H₀: μ₁ = μ₂ = μ₃`. Simulamos la masa corporal de 3 especies de pingüinos. `F = MS_between / MS_within`.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

adelie    = rng.normal(3700, 460, 150)
gentoo    = rng.normal(5080, 500, 120)
chinstrap = rng.normal(3730, 380, 68)
F, p = stats.f_oneway(adelie, gentoo, chinstrap)
k = 3
N = len(adelie) + len(gentoo) + len(chinstrap)
print(f"One-way ANOVA: F={F:.2f}  p={p:.2e}")
print(f"gl_between={k-1}  gl_within={N-k}")
assert p < 0.001

## 2. Supuestos (Levene) + effect size η²

Levene testea igualdad de varianzas; si rechaza, conviene Welch ANOVA. `η² = SS_between / SS_total` es la proporción de varianza explicada.

In [ ]:
lev = stats.levene(adelie, gentoo, chinstrap)
print(f"Levene (homocedasticidad): p={lev.pvalue:.4f}")

grupos = [adelie, gentoo, chinstrap]
alld = np.concatenate(grupos)
gm = alld.mean()
ss_between = sum(len(g) * (g.mean() - gm) ** 2 for g in grupos)
ss_total = ((alld - gm) ** 2).sum()
eta2 = ss_between / ss_total
print(f"η² = {eta2:.3f}  (>0.14 = efecto grande)")
assert eta2 > 0.5

## 3. Post-hoc Tukey HSD

ANOVA solo dice que **al menos una** media difiere. Tukey identifica qué pares, controlando el family-wise error rate.

In [ ]:
df = pd.DataFrame({
    "mass": alld,
    "species": ["Adelie"]*len(adelie) + ["Gentoo"]*len(gentoo) + ["Chinstrap"]*len(chinstrap),
})
tuk = pairwise_tukeyhsd(df["mass"], df["species"], alpha=0.05)
print(tuk)
# Gentoo se separa de las otras dos; Adelie y Chinstrap tienen masa casi igual
assert tuk.reject.sum() >= 2, "Gentoo debería diferir de Adelie y de Chinstrap" 

## 4. ANOVA de dos vías con interacción

Dos factores (dieta × ejercicio). La interacción responde: *¿el efecto de un factor depende del nivel del otro?* Ajustamos con OLS y leemos la tabla ANOVA tipo II.

In [ ]:
nrep = 40
rows = []
for dieta in ("A", "B"):
    for ej in ("no", "si"):
        base = 10 + 2*(dieta == "B") + 1.5*(ej == "si") + 3.0*((dieta == "B") and (ej == "si"))
        for v in rng.normal(base, 2.0, nrep):
            rows.append((dieta, ej, v))
d2 = pd.DataFrame(rows, columns=["dieta", "ejercicio", "y"])

model = smf.ols("y ~ C(dieta) * C(ejercicio)", data=d2).fit()
tab = anova_lm(model, typ=2)
print(tab)
p_inter = tab.loc["C(dieta):C(ejercicio)", "PR(>F)"]
print(f"\np interacción = {p_inter:.2e}")
assert p_inter < 0.05, "la interacción simulada debe ser significativa" 

## 5. Interaction plot

Si las líneas **no** son paralelas, hay interacción visual: el efecto del ejercicio es mayor con la dieta B.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for ej, style in zip(("no", "si"), ("o-", "s--")):
    sub = d2[d2.ejercicio == ej].groupby("dieta")["y"].mean()
    ax.plot(sub.index, sub.values, style, label=f"ejercicio={ej}")
ax.set_title("Interaction plot: líneas no paralelas => interacción")
ax.set_xlabel("dieta"); ax.set_ylabel("y medio"); ax.legend()
plt.tight_layout(); plt.show()

## Ejercicios

1. Recalculá el ANOVA one-way como `smf.ols("mass ~ C(species)")` + `anova_lm` y verificá que F y p coinciden con `f_oneway`.
2. Calculá ω² (menos sesgado que η²): `ω² = (SS_between - (k-1)·MS_within) / (SS_total + MS_within)`.
3. Interpretá la tabla two-way: ¿son interpretables los efectos principales dado que la interacción es significativa?

## Conclusiones

- ANOVA evita la inflación de α que produce hacer muchos t-tests por pares.
- Verificá homocedasticidad (Levene); si se viola, usá Welch ANOVA.
- Tukey HSD dice **qué** pares difieren controlando el error de familia.
- En two-way, interpretá primero la interacción: si es significativa, los efectos principales no se leen por separado.